In [8]:
import pandas as pd
import requests


YELP_API_KEY = "FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx"

# Definimos los parámetros de búsqueda
CIUDAD = "Boston"
URL_API = "https://api.yelp.com/v3/businesses/search"

headers = {
    "Authorization": f"Bearer {YELP_API_KEY}"
}

# Filtramos por restaurantes 
params = {
    "location": CIUDAD,
    "categories": "restaurants", 
    "limit": 50  # Traemos el límite máximo permitido por petición para tener una buena muestra
}

print(f"--- Conectando a la API de Yelp Fusion para la ciudad de: {CIUDAD} ---")

try:
    response = requests.get(URL_API, headers=headers, params=params)
    
    if response.status_code == 200:
        data_json = response.json()
        lista_negocios = data_json.get('businesses', [])
        print(f"✔ Conexión exitosa. Se extrajeron {len(lista_negocios)} negocios locales.")
        
    
        registros_limpios = []
        
        for biz in lista_negocios:
            # Procesamos las categorías que vienen en una lista interna del JSON
            categorias_lista = [c['title'] for c in biz.get('categories', [])]
            categoria_principal = categorias_lista[0] if categorias_lista else "General"
            
            # Extraemos exactamente los campos solicitados por la consigna
            registros_limpios.append({
                "ciudad_residencia": CIUDAD,              # Clave idéntica para hacer el Merge posterior
                "yelp_nombre_negocio": biz.get('name'),
                "yelp_categoria": categoria_principal,
                "yelp_calificacion": biz.get('rating'),     # Calificación (Rating de estrellas)
                "yelp_numero_reseñas": biz.get('review_count'), # Número de reseñas
                "yelp_precio": biz.get('price', 'N/A'),     # Rango de precio (e.g., $$, $$$)
                "yelp_telefono": biz.get('phone', 'N/A')
            })
        
        # Creamos el DataFrame exclusivo de Yelp
        df_yelp = pd.DataFrame(registros_limpios)
        print("\n--- Estructura de los datos extraídos de Yelp ---")
        print(df_yelp.head(5))
        
        
        print("\n--- Cargando base de clientes local para la integración ---")
        df_clientes = pd.read_csv('C:/Users/Jere/Desktop/Henry/base_datos_restaurantes_USA_v2.csv') 
        df_clientes_boston = df_clientes[df_clientes['ciudad_residencia'] == CIUDAD].copy()
        
        # Realizamos un Inner Join utilizando la columna 'ciudad_residencia' como puente común
        df_integrado_final = pd.merge(df_clientes_boston, df_yelp, on='ciudad_residencia', how='inner')
        
        print(f"✔ ¡Integración completada con éxito!")
        print(f"Dimensiones del dataset final unificado: {df_integrado_final.shape}")
        
        # Guardamos el resultado del avance en un nuevo CSV para el reporte final
        df_integrado_final.to_csv('clientes_boston_enriquecido_yelp.csv', index=False)
        print("✔ Dataset enriquecido guardado localmente como 'clientes_boston_enriquecido_yelp.csv'.")

    else:
        print(f"❌ Error de autenticación o parámetros ({response.status_code}).")
        print("Respuesta de la API:", response.text)

except Exception as e:
    print(f"❌ Falló la conexión de red con la API: {e}")

print (df_integrado_final)

--- Conectando a la API de Yelp Fusion para la ciudad de: Boston ---
✔ Conexión exitosa. Se extrajeron 50 negocios locales.

--- Estructura de los datos extraídos de Yelp ---
  ciudad_residencia           yelp_nombre_negocio yelp_categoria  \
0            Boston            The Friendly Toast           Bars   
1            Boston                   Carmelina's        Italian   
2            Boston              Boston Sail Loft           Bars   
3            Boston                 The Salty Pig   New American   
4            Boston  Mike & Patty's - Bay Village     Sandwiches   

   yelp_calificacion  yelp_numero_reseñas yelp_precio yelp_telefono  
0                4.3                 3567          $$  +16172319562  
1                4.5                 4520         $$$  +16177420020  
2                4.4                 2186          $$  +16172277280  
3                4.2                 2105          $$  +16175366200  
4                4.4                 2274          $$  +1617423344